# Partie 4 - Etape 1 (Food101) : Transfer Learning Séquentiel

Reprise stricte du protocole du notebook `04_strategy1_transfer_learning.ipynb`
mais sur le **dataset Food-101 (10 classes)** préparé par `14_Prepare_Food101_Dataset.ipynb`.

## Objectif
- Phase A : pré-entraîner un ResNet-18 sur le domaine BF (images dégradées).
- Phase B : fine-tuning sur le domaine HF (images propres).
- Évaluer sur test HF, test BF et précision mixte.

## Sorties attendues
- Métriques finales (HF, BF, Mixte, temps, coût) pour Food101.
- Courbes de loss/accuracy par phase.
- Sauvegarde JSON + poids du modèle dans `results/Food101/`.

In [ ]:
import os, json, time, random, shutil
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models
from torch.amp import autocast, GradScaler
import matplotlib.pyplot as plt

try:
    import wandb
    _WANDB_AVAILABLE = True
except ImportError:
    _WANDB_AVAILABLE = False
    print('wandb non installe : pip install wandb pour activer le tracking.')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Appareil: {device}')

import sys, importlib
_SRC_CANDIDATES = ['/content/drive/MyDrive/UTBM_PF22/src', '../src', 'src', './src']
for _p in _SRC_CANDIDATES:
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.insert(0, _p)
import degradation
importlib.reload(degradation)
from degradation import (DegradedDataset, hf_transform, clean_tensor_transform,
                          normalize_transform, IMAGENET_MEAN, IMAGENET_STD)
print('Module degradation charge depuis:', os.path.dirname(degradation.__file__))
import cost
importlib.reload(cost)
from cost import data_cost, unit_cost
import env_config
importlib.reload(env_config)

## 1) Montage Drive et chemins de travail (Food101)
Différences avec `04` :
- `DATASET_NAME = 'Food101'` (10 classes de plats)
- `RESULTS_DIR` écrit dans `results/Food101/` pour ne pas écraser les résultats précédents.

In [ ]:
DATASET_NAME = 'Food101'
BASE_DIR = env_config.ensure_dataset_ready(DATASET_NAME)
RESULTS_DIR = env_config.results_dir(DATASET_NAME)
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Dataset    :', DATASET_NAME)
print('BASE_DIR   :', BASE_DIR)
print('RESULTS_DIR:', RESULTS_DIR)

## 2) Datasets et transforms
Food101 a **10 classes** — même protocole de dégradation que les autres datasets.

In [ ]:
COST_HF = 10
COST_BF = 1
BATCH_SIZE = 64
NUM_WORKERS = min(8, (os.cpu_count() or 2))
PRETRAIN_EPOCHS = 10
FINETUNE_EPOCHS = 10
LR_PRETRAIN = 1e-3
LR_FINETUNE = 3e-4
HF_TRAIN_RATIO = 0.8

transform_hf    = hf_transform()
transform_clean = clean_tensor_transform()

required_dirs = [
    os.path.join(BASE_DIR, 'train', 'BF'),
    os.path.join(BASE_DIR, 'train', 'HF'),
    os.path.join(BASE_DIR, 'test'),
]
missing = [p for p in required_dirs if not os.path.isdir(p)]
if missing:
    raise FileNotFoundError('Structure dataset invalide. Dossiers manquants: ' + ' | '.join(missing))

dataset_bf_train = DegradedDataset(
    datasets.ImageFolder(os.path.join(BASE_DIR, 'train/BF'), transform=transform_clean),
    seeded=True)
dataset_hf_full  = datasets.ImageFolder(os.path.join(BASE_DIR, 'train/HF'), transform=transform_hf)
dataset_test_hf  = datasets.ImageFolder(os.path.join(BASE_DIR, 'test'), transform=transform_hf)
dataset_test_bf  = DegradedDataset(
    datasets.ImageFolder(os.path.join(BASE_DIR, 'test'), transform=transform_clean),
    seeded=True)

NUM_CLASSES = len(dataset_hf_full.classes)
print(f'Classes ({NUM_CLASSES}): {dataset_hf_full.classes}')

hf_train_size = int(HF_TRAIN_RATIO * len(dataset_hf_full))
hf_val_size   = len(dataset_hf_full) - hf_train_size
dataset_hf_train, dataset_hf_val = random_split(
    dataset_hf_full, [hf_train_size, hf_val_size],
    generator=torch.Generator().manual_seed(SEED))

loader_bf_train = DataLoader(dataset_bf_train, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
loader_hf_train = DataLoader(dataset_hf_train, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True)
loader_hf_val   = DataLoader(dataset_hf_val,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
loader_test_hf  = DataLoader(dataset_test_hf,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)
loader_test_bf  = DataLoader(dataset_test_bf,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train BF: {len(dataset_bf_train)} | Train HF: {len(dataset_hf_train)} | Val HF: {len(dataset_hf_val)}')
print(f'Test HF/BF: {len(dataset_test_hf)} | NUM_WORKERS: {NUM_WORKERS}')

## 3) Utilitaires de modélisation

In [ ]:
def create_model(num_classes=10):
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(device)

def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total_correct, total_samples = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            with autocast(device_type='cuda', enabled=(device.type == 'cuda')):
                logits = model(x)
                loss   = criterion(logits, y)
            total_loss    += loss.item() * x.size(0)
            total_correct += (logits.argmax(dim=1) == y).sum().item()
            total_samples += x.size(0)
    return total_loss / total_samples, 100.0 * total_correct / total_samples

def make_grad_scaler():
    try:
        return GradScaler('cuda', enabled=(device.type == 'cuda'))
    except TypeError:
        return GradScaler(enabled=(device.type == 'cuda'))

def train_phase(model, train_loader, val_loader, epochs, lr, phase_name,
                use_wandb=False, cost_per_epoch_CA=0,
                global_epoch_offset=0, cumulative_cost_start=0):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scaler    = make_grad_scaler()
    hist      = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    phase_start = time.time()
    log_every   = 10
    for epoch in range(1, epochs + 1):
        model.train()
        running_loss, total = 0.0, 0
        epoch_start = time.time()
        for batch_idx, (x, y) in enumerate(train_loader, start=1):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad(set_to_none=True)
            with autocast(device_type='cuda', enabled=(device.type == 'cuda')):
                logits = model(x)
                loss   = criterion(logits, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            running_loss += loss.item() * x.size(0)
            total        += x.size(0)
            if batch_idx % log_every == 0 or batch_idx == 1 or batch_idx == len(train_loader):
                elapsed = time.time() - epoch_start
                bps     = batch_idx / elapsed if elapsed > 0 else 0.0
                eta_sec = (len(train_loader) - batch_idx) / bps if bps > 0 else 0.0
                print(f'[{phase_name}] Epoch {epoch}/{epochs} | Batch {batch_idx}/{len(train_loader)} | '
                      f'loss={loss.item():.4f} | {bps:.2f} b/s | ETA={eta_sec/60:.1f} min')
        train_loss        = running_loss / total
        val_loss, val_acc = evaluate(model, val_loader, criterion)
        epoch_min         = (time.time() - epoch_start) / 60.0
        hist['train_loss'].append(train_loss)
        hist['val_loss'].append(val_loss)
        hist['val_acc'].append(val_acc)
        print(f'[{phase_name}] Epoch {epoch}/{epochs} en {epoch_min:.2f} min | '
              f'train={train_loss:.4f} val={val_loss:.4f} acc={val_acc:.2f}%')
        if use_wandb and _WANDB_AVAILABLE:
            cumulative_cost = cumulative_cost_start + cost_per_epoch_CA * epoch
            wandb.log({'epoch': global_epoch_offset + epoch, 'phase': phase_name,
                       'train/loss': train_loss, 'val/loss': val_loss,
                       'val/accuracy': val_acc, 'cumulative_cost_CA': cumulative_cost,
                       f'{phase_name}/train_loss': train_loss,
                       f'{phase_name}/val_loss': val_loss,
                       f'{phase_name}/val_accuracy': val_acc})
    elapsed_min = (time.time() - phase_start) / 60.0
    print(f'[{phase_name}] Termine en {elapsed_min:.2f} minutes')
    return hist, elapsed_min

## 4) Exécution Stratégie 1 (Food101) : BF → HF

In [ ]:
import statistics

SEEDS          = [42, 1, 2]
USE_WANDB      = True
WANDB_PROJECT  = 'PF22-MultiFidelity'
WANDB_RUN_NAME = 'Strategie1_TransferLearning'

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def run_seed(seed):
    set_all_seeds(seed)
    hf_train_size = int(HF_TRAIN_RATIO * len(dataset_hf_full))
    hf_val_size   = len(dataset_hf_full) - hf_train_size
    ds_hf_train, ds_hf_val = random_split(
        dataset_hf_full, [hf_train_size, hf_val_size],
        generator=torch.Generator().manual_seed(seed))
    ld_bf_train = DataLoader(dataset_bf_train, batch_size=BATCH_SIZE, shuffle=True,
                             num_workers=NUM_WORKERS, pin_memory=True)
    ld_hf_train = DataLoader(ds_hf_train, batch_size=BATCH_SIZE, shuffle=True,
                             num_workers=NUM_WORKERS, pin_memory=True)
    ld_hf_val   = DataLoader(ds_hf_val, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=True)
    track = USE_WANDB and _WANDB_AVAILABLE
    if track:
        wandb.init(project=WANDB_PROJECT, name=f'{WANDB_RUN_NAME}_seed{seed}',
                   tags=['strategy1', 'transfer_learning', DATASET_NAME, f'seed_{seed}'],
                   reinit=True,
                   config={'strategy': 'transfer_learning_sequentiel', 'dataset': DATASET_NAME,
                           'seed': seed, 'pretrain_epochs': PRETRAIN_EPOCHS,
                           'finetune_epochs': FINETUNE_EPOCHS, 'lr_pretrain': LR_PRETRAIN,
                           'lr_finetune': LR_FINETUNE, 'batch_size': BATCH_SIZE,
                           'architecture': 'resnet18', 'num_classes': NUM_CLASSES})
    model = create_model(num_classes=NUM_CLASSES)
    cost_pe_pre = len(dataset_bf_train) * COST_BF
    cost_pe_ft  = len(ds_hf_train) * COST_HF
    hist_pre, t_pre = train_phase(
        model, ld_bf_train, ld_hf_val, PRETRAIN_EPOCHS, LR_PRETRAIN, 'Pretrain_BF',
        use_wandb=track, cost_per_epoch_CA=cost_pe_pre,
        global_epoch_offset=0, cumulative_cost_start=0)
    hist_ft, t_ft = train_phase(
        model, ld_hf_train, ld_hf_val, FINETUNE_EPOCHS, LR_FINETUNE, 'Finetune_HF',
        use_wandb=track, cost_per_epoch_CA=cost_pe_ft,
        global_epoch_offset=PRETRAIN_EPOCHS,
        cumulative_cost_start=cost_pe_pre * PRETRAIN_EPOCHS)
    crit = nn.CrossEntropyLoss()
    _, hf = evaluate(model, loader_test_hf, crit)
    _, bf = evaluate(model, loader_test_bf, crit)
    mix   = (hf + bf) / 2.0
    if track:
        wandb.log({'test/accuracy_HF': hf, 'test/accuracy_BF': bf, 'test/accuracy_Mixte': mix})
        wandb.finish()
    print(f'[seed {seed}] HF={hf:.2f} BF={bf:.2f} Mixte={mix:.2f} (t={t_pre + t_ft:.1f} min)')
    return {'seed': seed, 'accuracy_HF': hf, 'accuracy_BF': bf, 'accuracy_Mixte': mix,
            'time_min': t_pre + t_ft, 'n_hf_train': len(ds_hf_train),
            'hist_pre': hist_pre, 'hist_ft': hist_ft, 'model': model}

per_seed = [run_seed(s) for s in SEEDS]
model    = per_seed[0]['model']
hist_pre = per_seed[0]['hist_pre']
hist_ft  = per_seed[0]['hist_ft']

## 5) Évaluation finale, coût, sauvegarde et visualisation

In [ ]:
def _agg(key):
    vals = [r[key] for r in per_seed]
    m = statistics.mean(vals)
    s = statistics.stdev(vals) if len(vals) > 1 else 0.0
    return m, s, vals

hf_m, hf_s, hf_all = _agg('accuracy_HF')
bf_m, bf_s, bf_all = _agg('accuracy_BF')
mx_m, mx_s, mx_all = _agg('accuracy_Mixte')
tt_m, tt_s, _      = _agg('time_min')

n_hf_train   = per_seed[0]['n_hf_train']
data_cost_CA = data_cost(n_hf=len(dataset_hf_full), n_bf=len(dataset_bf_train))
compute_seen = len(dataset_bf_train) * PRETRAIN_EPOCHS + n_hf_train * FINETUNE_EPOCHS
total_cost   = (len(dataset_bf_train) * COST_BF * PRETRAIN_EPOCHS
                + n_hf_train * COST_HF * FINETUNE_EPOCHS)

results = {
    'strategy': 'transfer_learning_sequentiel', 'dataset': DATASET_NAME,
    'pipeline': 'BF_pretraining_then_HF_finetuning',
    'pretrain_epochs': PRETRAIN_EPOCHS, 'finetune_epochs': FINETUNE_EPOCHS,
    'lr_pretrain': LR_PRETRAIN, 'lr_finetune': LR_FINETUNE, 'seeds': SEEDS,
    'num_classes': NUM_CLASSES,
    'data_cost_CA': float(data_cost_CA), 'compute_images_seen': int(compute_seen),
    'total_cost_CA': float(total_cost),
    'n_hf_pool': int(len(dataset_hf_full)), 'n_bf_pool': int(len(dataset_bf_train)),
    'hf_images_seen': int(n_hf_train * FINETUNE_EPOCHS),
    'bf_images_seen': int(len(dataset_bf_train) * PRETRAIN_EPOCHS),
    'accuracy_HF': hf_m, 'accuracy_HF_std': hf_s, 'accuracy_HF_seeds': hf_all,
    'accuracy_BF': bf_m, 'accuracy_BF_std': bf_s, 'accuracy_BF_seeds': bf_all,
    'accuracy_Mixte': mx_m, 'accuracy_Mixte_std': mx_s, 'accuracy_Mixte_seeds': mx_all,
    'total_time_min': float(tt_m), 'total_time_min_std': float(tt_s),
    'per_seed': [{k: v for k, v in r.items() if k not in ('model', 'hist_pre', 'hist_ft')}
                 for r in per_seed],
}

json_path = os.path.join(RESULTS_DIR, 'results_strategy1_transfer_learning.json')
pth_path  = os.path.join(RESULTS_DIR, 'model_strategy1_transfer_learning.pth')
png_path  = os.path.join(RESULTS_DIR, 'strategy1_transfer_learning_curves.png')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
torch.save(model.state_dict(), pth_path)

print(f'--- RESULTATS ({len(SEEDS)} seeds) - {DATASET_NAME} - Strategie 1 ---')
print(f'Test HF    : {hf_m:.2f} +/- {hf_s:.2f} %')
print(f'Test BF    : {bf_m:.2f} +/- {bf_s:.2f} %')
print(f'Test Mixte : {mx_m:.2f} +/- {mx_s:.2f} %')
print(f'Cout donnees: {data_cost_CA:.0f} | calcul: {compute_seen} | total: {total_cost:.0f} CA')
print('JSON:', json_path)

x_pre = np.arange(1, PRETRAIN_EPOCHS + 1)
x_ft  = np.arange(PRETRAIN_EPOCHS + 1, PRETRAIN_EPOCHS + FINETUNE_EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Strategie 1 (BF -> HF) - {DATASET_NAME} (seed {SEEDS[0]})', fontsize=14)
axes[0].plot(x_pre, hist_pre['train_loss'], marker='o', label='Train loss - Pretrain BF')
axes[0].plot(x_pre, hist_pre['val_loss'],   marker='o', linestyle='--', label='Val loss HF - Pretrain BF')
axes[0].plot(x_ft,  hist_ft['train_loss'],  marker='o', label='Train loss - Finetune HF')
axes[0].plot(x_ft,  hist_ft['val_loss'],    marker='o', linestyle='--', label='Val loss HF - Finetune HF')
axes[0].axvline(PRETRAIN_EPOCHS + 0.5, color='gray', linestyle=':', label='Transition BF -> HF')
axes[0].set_title('Evolution de la loss')
axes[0].set_xlabel('Epoch globale')
axes[0].set_ylabel('Loss')
axes[0].grid(alpha=0.3)
axes[0].legend(fontsize=8)
axes[1].plot(x_pre, hist_pre['val_acc'], marker='o', label='Val acc HF - Pretrain BF')
axes[1].plot(x_ft,  hist_ft['val_acc'],  marker='o', label='Val acc HF - Finetune HF')
axes[1].axvline(PRETRAIN_EPOCHS + 0.5, color='gray', linestyle=':', label='Transition BF -> HF')
axes[1].set_title('Evolution de la precision (validation HF)')
axes[1].set_xlabel('Epoch globale')
axes[1].set_ylabel('Accuracy (%)')
axes[1].grid(alpha=0.3)
axes[1].legend(fontsize=8)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.savefig(png_path, dpi=180, bbox_inches='tight')
plt.show()
print('Figure:', png_path)

## 6) Comparaison avec les baselines Food101

In [ ]:
baseline_files = {
    'BL 1(HF)':            'results_baseline_HF.json',
    'BL 2(BF)':            'results_baseline_BF.json',
    'BL 3(MIXTE)':         'results_baseline_MIXTE.json',
    'Strategie 1(BF->HF)': 'results_strategy1_transfer_learning.json',
}

rows = []
for name, fn in baseline_files.items():
    p = os.path.join(RESULTS_DIR, fn)
    if os.path.exists(p):
        with open(p, 'r', encoding='utf-8') as f:
            d = json.load(f)
        rows.append((
            name,
            d.get('accuracy_HF', float('nan')),
            d.get('accuracy_BF', float('nan')),
            d.get('accuracy_Mixte', float('nan')),
            d.get('total_cost_CA', float('nan')),
            d.get('training_time_sec', float('nan')) / 60.0 if 'training_time_sec' in d
            else d.get('total_time_min', float('nan')),
        ))

if not rows:
    print('Aucun fichier JSON trouve pour la comparaison dans', RESULTS_DIR)
else:
    header = f"{'Modele (Food101)':<24} {'HF%':>8} {'BF%':>8} {'Mixte%':>10} {'Cout(CA)':>12} {'Temps(min)':>12}"
    print(header)
    print('-' * len(header))
    for r in rows:
        print(f'{r[0]:<24} {r[1]:>8.2f} {r[2]:>8.2f} {r[3]:>10.2f} {r[4]:>12.0f} {r[5]:>12.2f}')